# Scientist #7: Christiaan Huygens — Wave Theory of Light (Cycles Turbo Dual-GPU Short)

**ZERO Quality Compromise — 6x Faster True Cycles Raytracing (~4-6 Minutes Total on Kaggle `GPU T4 x2`)!**
- **Full 30-Second Cinema Master**: Pre-mastered Kokoro multi-voice narration + 116 BPM Baroque Score + 24 Spatial SFX + Word-Synced Boxless Subtitles.
- **Cycles Turbo Optimizations**: Clean glass/brass (noisy caustics off + glossy filter), `RGB_ALBEDO` fast OIDN, `24fps` cinema rate (`720` frames), `75%` (`810x1440`) raytrace -> razor-sharp Lanczos `1080x1920` upscale, zero-stall `JPEG-98` output, and `OMP_NUM_THREADS=2` per GPU worker!

### Quick Kaggle Setup:
1. Right panel -> **Session options** -> **Accelerator: `GPU T4 x2`** & **Internet: `ON`**.
2. Run **Cell 1 -> Cell 2 -> Cell 3** (10-second **6-Beat Visual Preview Grid**!).
3. Run **Cell 4 -> Cell 5** (~5 min) -> download **`/kaggle/working/christiaan_huygens_wave_light_short.mp4`**!

In [ ]:
# 1. Verify NVIDIA GPU(s) & Clone Latest V2 Cycles Turbo Repo
!nvidia-smi
!rm -rf huygens-wave-light-3d
!git clone https://github.com/ssghtc002-sudo/huygens-wave-light-3d.git
%cd huygens-wave-light-3d
!ls -lh audio/output/master_audio.wav subtitles/subtitles.ass

In [ ]:
# 2. Download & Extract Blender 4.2.3 LTS Headless (~25 sec, cached in /tmp/blender)
import os
if not os.path.exists('/tmp/blender/blender'):
    !wget -q https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz -O /tmp/blender.tar.xz
    !mkdir -p /tmp/blender && tar -xf /tmp/blender.tar.xz -C /tmp/blender --strip-components=1
!/tmp/blender/blender --version

In [ ]:
# 3. Fast 6-Beat Visual QA Preview (~8 seconds!) + Live Speed Check
import time, glob, os, base64
from IPython.display import display, HTML

t0 = time.time()
!CUDA_VISIBLE_DEVICES=0 OMP_NUM_THREADS=2 /tmp/blender/blender --background --python src/build_huygens_scene.py -- --qa-only --fps 24 --samples 12 --scale 75
dt = time.time() - t0
print(f'\n[QA Speed] 6 Beat frames rendered in {dt:.1f}s (~{dt/6:.2f}s/frame -> Est. Full 720-frame Dual-GPU time: ~{(dt/6)*360/60:.1f} min!)')

imgs = sorted(glob.glob('qa_frames/qa_beat*.jpg'))
cards = []
for p in imgs:
    b64 = base64.b64encode(open(p, 'rb').read()).decode()
    cards.append(f'<div style="display:inline-block;margin:6px;text-align:center;"><img src="data:image/jpeg;base64,{b64}" width="210"/><br><small>{os.path.basename(p)}</small></div>')
display(HTML(''.join(cards)))

In [ ]:
# 4. Parallel Dual-GPU Cycles Turbo Render (720 frames @ 24fps, 12 spp, 75% -> 1080p Lanczos -> ~4-6 min on T4 x2!)
# Splits 1..360 on GPU 0 and 361..720 on GPU 1 with isolated CPU threads (OMP_NUM_THREADS=2) so zero denoise contention!
import subprocess, time, glob

TOTAL = 720
gpu_count = int(subprocess.check_output('nvidia-smi --list-gpus | wc -l', shell=True).decode().strip())
print(f'Detected {gpu_count} NVIDIA GPU(s)! Starting Cycles Turbo production render ({TOTAL} frames @ 24fps)...')
t0 = time.time()

if gpu_count >= 2:
    p0 = subprocess.Popen(
        'CUDA_VISIBLE_DEVICES=0 OMP_NUM_THREADS=2 /tmp/blender/blender --background --python src/build_huygens_scene.py -- --start-frame 1 --end-frame 360 --fps 24 --samples 12 --scale 75',
        shell=True, stdout=open('/tmp/gpu0.log', 'w'), stderr=subprocess.STDOUT
    )
    p1 = subprocess.Popen(
        'CUDA_VISIBLE_DEVICES=1 OMP_NUM_THREADS=2 /tmp/blender/blender --background --python src/build_huygens_scene.py -- --start-frame 361 --end-frame 720 --fps 24 --samples 12 --scale 75',
        shell=True, stdout=open('/tmp/gpu1.log', 'w'), stderr=subprocess.STDOUT
    )
    while p0.poll() is None or p1.poll() is None:
        done = len(glob.glob('render_frames/frame_*.jpg'))
        elapsed = time.time() - t0
        rate = done / max(1.0, elapsed)
        eta = (TOTAL - done) / max(0.01, rate) if done > 2 else 0
        print(f'\r[Dual-T4 Turbo] {done}/{TOTAL} frames ({done*100/TOTAL:.1f}%) | Elapsed: {elapsed/60:.1f}m | ETA: {eta/60:.1f}m   ', end='', flush=True)
        time.sleep(3)
    print()
    if p0.returncode != 0 or p1.returncode != 0:
        print('--- GPU 0 LOG TAIL ---\n' + ''.join(open('/tmp/gpu0.log').readlines()[-25:]))
        print('--- GPU 1 LOG TAIL ---\n' + ''.join(open('/tmp/gpu1.log').readlines()[-25:]))
        raise RuntimeError(f'Worker error: p0={p0.returncode} p1={p1.returncode}')
else:
    !OMP_NUM_THREADS=2 /tmp/blender/blender --background --python src/build_huygens_scene.py -- --start-frame 1 --end-frame 720 --fps 24 --samples 12 --scale 75

rendered = len(glob.glob('render_frames/frame_*.jpg'))
print(f'Render complete: {rendered}/{TOTAL} frames in {(time.time()-t0)/60:.1f} min!')

In [ ]:
# 5. Lanczos Crisp 1080x1920 Upscale + Mastered 30s Kokoro Audio + Word-Synced Boxless Subtitles -> Final MP4!
!mkdir -p output
!ffmpeg -y -framerate 24 -i render_frames/frame_%04d.jpg -i audio/output/master_audio.wav \
  -vf "scale=1080:1920:flags=lanczos,unsharp=3:3:0.35:3:3:0.0,ass=subtitles/subtitles.ass" \
  -c:v libx264 -profile:v high -crf 15 -pix_fmt yuv420p \
  -c:a aac -b:a 192k -shortest -movflags +faststart output/christiaan_huygens_wave_light_short.mp4 -loglevel warning

# Copy directly to /kaggle/working/ so it appears immediately in Kaggle Output panel for 1-tap download!
import os, shutil
out_mp4 = 'output/christiaan_huygens_wave_light_short.mp4'
if os.path.isdir('/kaggle/working'):
    shutil.copy2(out_mp4, '/kaggle/working/christiaan_huygens_wave_light_short.mp4')
    print('Copied to /kaggle/working/christiaan_huygens_wave_light_short.mp4 — Ready in Kaggle Output Panel!')

!ffprobe -v error -select_streams v:0 -show_entries stream=width,height,r_frame_rate,nb_frames,duration -of compact output/christiaan_huygens_wave_light_short.mp4
!ls -lh output/christiaan_huygens_wave_light_short.mp4

try:
    from google.colab import files
    files.download(out_mp4)
except ImportError:
    from IPython.display import Video, display
    display(Video(out_mp4, embed=False, width=320))